# Temperature and Sampling

At each generation step, the model produces scores (logits) for possible next tokens. Temperature changes the distribution over those tokens; top-k and top-p remove candidates. This notebook simulates that selection process locally, without a model API or extra packages.

By the end, compare the controls, explain why temperature 0 is not a provider-wide determinism guarantee, and choose settings for ideation, extraction, and code.

## 1. From logits to probabilities

Softmax turns scores into probabilities. Temperature $T$ divides each logit before softmax: $p_i = \exp(z_i/T) / \sum_j \exp(z_j/T)$. Lower positive temperatures sharpen the distribution; higher temperatures flatten it. The ranking is unchanged. In practice, temperature zero usually means greedy argmax decoding, not literal division by zero.

In [1]:
import math
import random

logits = {"maple": 2.4, "cedar": 1.7, "willow": 0.9, "cactus": 0.2}

def softmax(scores, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be positive; use greedy_token for temperature 0")
    scaled = {token: score / temperature for token, score in scores.items()}
    largest = max(scaled.values())
    weights = {token: math.exp(score - largest) for token, score in scaled.items()}
    total = sum(weights.values())
    return {token: weight / total for token, weight in weights.items()}

def greedy_token(scores):
    return max(scores, key=scores.get)

for temperature in (0.5, 1.0, 1.8):
    probabilities = softmax(logits, temperature)
    print(f"T={temperature}: {probabilities}")

assert math.isclose(sum(softmax(logits, 0.5).values()), 1.0)
assert softmax(logits, 0.5)["maple"] > softmax(logits, 1.8)["maple"]
assert greedy_token(logits) == "maple"

T=0.5: {'maple': 0.7641396171947151, 'cedar': 0.18843450962771816, 'willow': 0.038044271363866664, 'cactus': 0.009381601813700124}
T=1.0: {'maple': 0.5462932678293861, 'cedar': 0.27128120836425756, 'willow': 0.12189450433877999, 'cactus': 0.060531019467576365}
T=1.8: {'maple': 0.41545792389268554, 'cedar': 0.28160136007272174, 'willow': 0.1805572694338312, 'cactus': 0.12238344660076153}


## 2. Crop candidates: top-k and top-p

Top-k retains at most a fixed number of highest-probability tokens. Top-p (nucleus sampling) retains the smallest highest-probability prefix whose cumulative probability reaches the chosen threshold. Both renormalize the retained probabilities. This example applies at most one filter at a time to keep their effects easy to see.

In [2]:
def filter_candidates(probabilities, top_k=None, top_p=None):
    ranked = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)
    if top_k is not None:
        if top_k < 1:
            raise ValueError("top_k must be at least 1")
        ranked = ranked[:top_k]
    elif top_p is not None:
        if not 0 < top_p <= 1:
            raise ValueError("top_p must be in (0, 1]")
        kept = []
        cumulative = 0.0
        for item in ranked:
            kept.append(item)
            cumulative += item[1]
            if cumulative >= top_p:
                break
        ranked = kept
    total = sum(probability for _, probability in ranked)
    return {token: probability / total for token, probability in ranked}

base = softmax(logits, temperature=1.0)
top_k_probs = filter_candidates(base, top_k=2)
top_p_probs = filter_candidates(base, top_p=0.8)
print("base:  ", base)
print("top-k:", top_k_probs)
print("top-p:", top_p_probs)

assert len(top_k_probs) <= 2
assert math.isclose(sum(top_k_probs.values()), 1.0)
assert math.isclose(sum(top_p_probs.values()), 1.0)
assert sum(base[token] for token in top_p_probs) >= 0.8

base:   {'maple': 0.5462932678293861, 'cedar': 0.27128120836425756, 'willow': 0.12189450433877999, 'cactus': 0.060531019467576365}
top-k: {'maple': 0.6681877721681662, 'cedar': 0.3318122278318339}
top-p: {'maple': 0.6681877721681662, 'cedar': 0.3318122278318339}


## 3. Draw a token

Sampling makes a random draw from the retained distribution. An early draw changes the generated context, so later distributions can diverge too. Fixing a local random seed makes this toy simulation repeatable; it does not make a remote provider's serving stack deterministic.

In [3]:
def sample_token(probabilities, rng):
    draw = rng.random()
    cumulative = 0.0
    for token, probability in probabilities.items():
        cumulative += probability
        if draw < cumulative:
            return token
    return next(reversed(probabilities))  # protects against floating-point rounding

rng = random.Random(7)
print([sample_token(base, rng) for _ in range(12)])

['maple', 'maple', 'cedar', 'maple', 'maple', 'maple', 'maple', 'maple', 'maple', 'maple', 'maple', 'maple']


## 4. Match randomness to the task

These are starting points, not universal settings. Ideation benefits from diverse candidates, so moderate temperature (for example, 0.7–1) can help. Extraction usually favors low temperature (0–0.2) plus a schema and validation. Code often favors low-to-moderate temperature (0–0.3), with tests as the correctness check. Low temperature reduces variation; it does not guarantee correctness.

Temperature 0 is usually greedy decoding, but exact reproducibility can still be affected by provider routing, model updates, numerical ties, or nondeterministic serving kernels. A seed, when supported, is not a cross-provider guarantee.

## 5. Where the controls live in API requests

Sampling options are JSON request-body fields, not HTTP headers. These are illustrative fragments only; they do not make network calls. Availability and constraints depend on endpoint and model. OpenAI Chat Completions has top-level `temperature` and `top_p`, but no standard `top_k` field. Anthropic Messages has top-level `temperature`, `top_p`, and `top_k`; consult its current docs for supported combinations and model-specific rules.

In [4]:
openai_chat_body = {
    "model": "<model>",
    "messages": [{"role": "user", "content": "Suggest plant shop names"}],
    "temperature": 0.8,
    "top_p": 0.95,
}

anthropic_messages_body = {
    "model": "<model>",
    "max_tokens": 100,
    "messages": [{"role": "user", "content": "Extract the invoice number"}],
    "temperature": 0.1,
    "top_k": 5,
}

print("OpenAI Chat Completions fields:", [key for key in openai_chat_body if key in {"temperature", "top_p", "top_k"}])
print("Anthropic Messages fields:", [key for key in anthropic_messages_body if key in {"temperature", "top_p", "top_k"}])

OpenAI Chat Completions fields: ['temperature', 'top_p']
Anthropic Messages fields: ['temperature', 'top_k']


## Practice

1. Change the logits and compare distributions at three temperatures. What stays the same, and what changes?
2. Try top-k values 1, 2, and 4. Then try top-p values 0.5 and 0.9. Explain why the retained top-p set can change size.
3. Choose temperature and any truncation setting for ideation, extraction, and code. Defend each choice, including what you would validate afterward.
4. Explain why a repeated prompt at temperature 0 may not produce identical responses across time or serving environments.
5. Point to the sampling fields in each request dictionary. Which API example can express top-k?